# Defense Ablation Study  Adversarial Training

**Goal:** Fine-tune the trained DeBERTa reward model on our adversarial defense dataset at three different mixing ratios to find the optimal trade-off between clean accuracy and adversarial robustness.

**Ablation Ratios:**
- 10% adversarial mix  Minimal defense
- 25% adversarial mix  Balanced defense (primary hypothesis)
- 50% adversarial mix  Aggressive defense


**Outputs:** Three model checkpoints saved to `/kaggle/working/models/ablation/`.

### 1. Environment Setup

In [ ]:
import os, sys, subprocess

# Install dependencies
subprocess.run(['pip', 'install', '-q', 'transformers', 'sentencepiece', 'tqdm'], check=True)

# Set up paths  Kaggle input datasets are mounted at /kaggle/input/
KAGGLE_INPUT = '/kaggle/input/datasets'
WORKING_DIR  = '/kaggle/working'

# Paths to the dataset files attached to this Kaggle notebook:
#   Dataset 1: sage-data       contains data/adv_training_pairs.json
#   Dataset 2: sage-model      contains baseline_epoch_3.pt
#   Dataset 3: sage-src        contains the src/ Python package
DATA_PATH    = f'{KAGGLE_INPUT}/sage-data/adv_training_pairs.json'
MODEL_PATH   = f'{KAGGLE_INPUT}/sage-model/baseline_epoch_3.pt'
SRC_PATH     = f'{KAGGLE_INPUT}/sage-src'

# Add src to path
import shutil
if not os.path.exists(f'{WORKING_DIR}/src'):
    shutil.copytree(SRC_PATH, f'{WORKING_DIR}/src')
sys.path.insert(0, WORKING_DIR)

# Output directories
os.makedirs(f'{WORKING_DIR}/models/ablation', exist_ok=True)
os.makedirs(f'{WORKING_DIR}/results', exist_ok=True)

print('Environment ready.')
print(f'Data: {DATA_PATH}')
print(f'Model: {MODEL_PATH}')


### 2. Verify GPU

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be very slow on CPU!')

### 3. Run Ablation Study

Trains three models back-to-back. Each model is initialized from the same base checkpoint so the only variable is the adversarial mixing ratio.

In [ ]:
from src.robustness.adversarial_training import run_ablation_study

# Override paths for Kaggle environment
import src.robustness.adversarial_training as adv_train
adv_train.ADV_DATA_PATH  = DATA_PATH
adv_train.BASE_CHECKPOINT = MODEL_PATH
adv_train.OUTPUT_DIR     = f'{WORKING_DIR}/models/ablation'
adv_train.RESULTS_DIR    = f'{WORKING_DIR}/results'

results = run_ablation_study(
    adv_ratios=[0.10, 0.25, 0.50],
    epochs=2,
    device=device
)

### 4. Training Summary

In [ ]:
import json

print('\n--- Ablation Training Summary ---')
print(f'{"Ratio":<10} {"Pairs":<8} {"Final Acc":<12} {"Checkpoint"}')
print('-' * 70)
for r in results:
    print(f"{r['adv_ratio']:.0%}"       .ljust(10) +
          str(r['n_pairs']).ljust(8) +
          f"{r['final_train_acc']:.4f}".ljust(12) +
          os.path.basename(r['checkpoint']))

# Save a copy of the results summary for download
results_path = f"{adv_train.RESULTS_DIR}/ablation_training_summary.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSummary saved to: {results_path}')
print('Download the /kaggle/working/ directory to get the checkpoints!')